# 0. Data Types

Every NumPy array holds elements of a single data type (its `dtype`). Picking the right dtype matters for memory use, speed and correctness, so it's worth getting comfortable with them before anything else.

In [1]:
import numpy as np
np.__version__

'2.5.3'

## The main families

| Kind | Types | Type codes |
|---|---|---|
| Signed integers | `int8`, `int16`, `int32`, `int64` | `i1`, `i2`, `i4`, `i8` |
| Unsigned integers | `uint8`, `uint16`, `uint32`, `uint64` | `u1`, `u2`, `u4`, `u8` |
| Floating point | `float16`, `float32`, `float64` | `f2`, `f4`, `f8` |
| Complex | `complex64`, `complex128` | `c8`, `c16` |
| Boolean | `bool` | `?` |
| Fixed-length bytes | `bytes_` | `S` (e.g. `S5`) |
| Fixed-length unicode | `str_` | `U` (e.g. `U10`) |
| Python objects | `object_` | `O` |

The number in the type name is the size in **bits**; the number in the type code is the size in **bytes**. So `float32` and `f4` are the same thing.

There's also `float128` / `longdouble`, but its actual precision depends on your platform (on Windows it's just `float64`), so don't rely on it for portable code.

## Choosing a dtype

You can pass the dtype as a name string, a type code string, or a NumPy type object. They all mean the same thing:

In [2]:
a = np.array([1, 2, 3], dtype='float32')
b = np.array([1, 2, 3], dtype='f4')
c = np.array([1, 2, 3], dtype=np.float32)
a.dtype, b.dtype, c.dtype

(dtype('float32'), dtype('float32'), dtype('float32'))

In [3]:
z = np.array([1+2j, 3-4j], dtype='c8')
z.dtype

dtype('complex64')

Type codes only work as **strings**. There's no `np.c8` or `np.f4` attribute:

In [4]:
try:
    np.array([1+2j, 3-4j], dtype=np.c8)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: module 'numpy' has no attribute 'c8'


If you don't specify a dtype, NumPy picks one from the data. Integers default to `int64` on most platforms and floats to `float64`:

In [5]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1.0, 2, 3]).dtype)
print(np.array([True, False]).dtype)
print(np.array(['a', 'bc']).dtype)

int64
float64
bool
<U2


## Type limits

`np.iinfo` and `np.finfo` tell you the range and precision of a type. Handy when you're deciding whether a smaller type is safe to use.

In [6]:
print(np.iinfo(np.int8))
print(np.iinfo(np.uint16).max)
print(np.finfo(np.float32).eps)   # smallest step away from 1.0

Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------

65535
1.1920929e-07


## Converting between types

`astype` gives you a **new** array with the requested dtype. The original isn't touched, so remember to assign the result.

In [7]:
arr = np.array([1, 2, 3], dtype='int16')
print('before:', arr.dtype)

arr = arr.astype(np.float32)
print('after: ', arr.dtype)

before: int16
after:  float32


Converting floats to integers truncates toward zero, it doesn't round:

In [8]:
np.array([1.7, -1.7, 2.5]).astype(int)

array([ 1, -1,  2])

### Watch out for overflow

Going down to a smaller type (say `int16` to `int8`) silently wraps values that don't fit. These bugs are really annoying to track down, so check ranges first.

In [9]:
arr = np.array([126, 127, 256], dtype='int16')
print('int16:', arr)

# int8 can only hold -128..127, so 256 wraps around to 0
print('int8: ', arr.astype('int8'))

int16: [126 127 256]
int8:  [126 127   0]


Arithmetic can overflow too. Here 200 + 100 doesn't fit in a `uint8`:

In [10]:
pixels = np.array([200, 250], dtype=np.uint8)
print(pixels + np.uint8(100))           # wraps around
print(pixels.astype(np.int16) + 100)    # upcast first if you need the real answer

[44 94]
[300 350]


## Type promotion

When you combine arrays of different types, NumPy promotes to a type that can hold both. Since NumPy 2.0, plain Python scalars are "weak": they adapt to the array's type instead of forcing an upcast.

In [11]:
i = np.array([1, 2, 3], dtype=np.int32)
f = np.array([0.5, 0.5, 0.5], dtype=np.float32)

print((i + f).dtype)        # int32 + float32 -> float64
print((f + 1.0).dtype)      # Python float doesn't upcast float32
print(np.result_type(np.int8, np.uint8))

float64
float32
int16


## Strings

String arrays are **fixed width**. `S3` means bytes of length 3, `U10` means unicode of length 10. If you don't give a length, NumPy uses the longest string in the data. Anything longer than the declared width gets cut off without warning.

In [12]:
print(np.array(['abc', 'defg'], dtype='S3'))
print(np.array(['abcd', 'efghi'], dtype='U3'))

[b'abc' b'def']
['abc' 'efg']


In [13]:
arr = np.array(['a', 'abc', 'abcd'])
print(arr, arr.dtype)    # <U4 because 'abcd' is the longest

['a' 'abc' 'abcd'] <U4


This also means assigning a longer string into an existing array truncates it:

In [14]:
arr[0] = 'something long'
arr

array(['some', 'abc', 'abcd'], dtype='<U4')

Since NumPy 2.0 there's also `np.dtypes.StringDType()`, a variable-length string type that avoids the truncation problem entirely:

In [15]:
arr = np.array(['a', 'abc'], dtype=np.dtypes.StringDType())
arr[0] = 'something long'
arr

array(['something long', 'abc'], dtype=StringDType())

The `np.char` / `np.strings` module has vectorized string functions:

In [16]:
names = np.array(['alice', 'bob', 'carol'])
print(np.strings.upper(names))
print(np.strings.str_len(names))

['ALICE' 'BOB' 'CAROL']
[5 3 5]


### What do `<` and `|` mean in `<U4` or `|S3`?

They describe byte order: `<` is little-endian, `>` big-endian, and `|` means it doesn't apply (single bytes). You rarely need to care unless you're reading raw binary files from another system.

- https://numpy.org/doc/stable/reference/generated/numpy.dtype.byteorder.html
- https://en.wikipedia.org/wiki/Endianness